In [1]:
OUTPUT_DIR = "../data/raw/ontime"

In [2]:
import requests
import os
import time

BASE_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

year, month = 2015, 1  # just ONE file for now

filename = os.path.join(OUTPUT_DIR, f"ontime_{year}_{month:02d}.zip")
url = BASE_URL.format(year=year, month=month)

print(f"Downloading {year}-{month:02d} from {url} ...")
r = requests.get(url, timeout=60)
print(f"Status code: {r.status_code}")

if r.status_code == 200:
    with open(filename, "wb") as f:
        f.write(r.content)
    print(f"Saved {filename} ({len(r.content) / 1_000_000:.1f} MB)")
else:
    print("Didn't work — paste me the status code and I'll help debug.")

Status code: 200
Saved ../data/raw/ontime\ontime_2015_01.zip (23.1 MB)


In [3]:
import zipfile
import pandas as pd

zip_path = os.path.join(OUTPUT_DIR, "ontime_2015_01.zip")

# List what's actually inside the zip first — TranStats sometimes bundles
# a README or a differently-named CSV, so don't assume the filename
with zipfile.ZipFile(zip_path) as z:
    print("Files inside the zip:")
    for name in z.namelist():
        print(" -", name)

Files inside the zip:
 - On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
 - readme.html


In [4]:
csv_name = "On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv"

with zipfile.ZipFile(zip_path) as z:
    with z.open(csv_name) as f:
        df = pd.read_csv(f, nrows=1000)  # just first 1000 rows for now — full file can wait

print(f"Shape (first 1000 rows only): {df.shape}")
print(f"\nTotal columns: {len(df.columns)}")
print("\nColumn names:")
for col in df.columns:
    print(" -", col)

Shape (first 1000 rows only): (1000, 110)

Total columns: 110

Column names:
 - Year
 - Quarter
 - Month
 - DayofMonth
 - DayOfWeek
 - FlightDate
 - Reporting_Airline
 - DOT_ID_Reporting_Airline
 - IATA_CODE_Reporting_Airline
 - Tail_Number
 - Flight_Number_Reporting_Airline
 - OriginAirportID
 - OriginAirportSeqID
 - OriginCityMarketID
 - Origin
 - OriginCityName
 - OriginState
 - OriginStateFips
 - OriginStateName
 - OriginWac
 - DestAirportID
 - DestAirportSeqID
 - DestCityMarketID
 - Dest
 - DestCityName
 - DestState
 - DestStateFips
 - DestStateName
 - DestWac
 - CRSDepTime
 - DepTime
 - DepDelay
 - DepDelayMinutes
 - DepDel15
 - DepartureDelayGroups
 - DepTimeBlk
 - TaxiOut
 - WheelsOff
 - WheelsOn
 - TaxiIn
 - CRSArrTime
 - ArrTime
 - ArrDelay
 - ArrDelayMinutes
 - ArrDel15
 - ArrivalDelayGroups
 - ArrTimeBlk
 - Cancelled
 - CancellationCode
 - Diverted
 - CRSElapsedTime
 - ActualElapsedTime
 - AirTime
 - Flights
 - Distance
 - DistanceGroup
 - CarrierDelay
 - WeatherDelay
 - NA

In [5]:
KEEP_COLS = [
    "FlightDate", "Year", "Quarter", "Month", "DayOfWeek",
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "OriginState", "Dest", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes", "DepDel15",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "Cancelled", "CancellationCode", "Diverted",
    "CRSElapsedTime", "ActualElapsedTime", "AirTime",
    "Distance", "DistanceGroup",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

In [6]:
with zipfile.ZipFile(zip_path) as z:
    with z.open(csv_name) as f:
        df = pd.read_csv(f, usecols=KEEP_COLS)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing values (top 10):\n{df.isnull().sum().sort_values(ascending=False).head(10)}")
print(f"\nSample rows:")
df.head(3)

Shape: (469968, 35)

Dtypes:
Year                                 int64
Quarter                              int64
Month                                int64
DayOfWeek                            int64
FlightDate                          object
Reporting_Airline                   object
Tail_Number                         object
Flight_Number_Reporting_Airline      int64
Origin                              object
OriginState                         object
Dest                                object
DestState                           object
CRSDepTime                           int64
DepTime                            float64
DepDelay                           float64
DepDelayMinutes                    float64
DepDel15                           float64
CRSArrTime                           int64
ArrTime                            float64
ArrDelay                           float64
ArrDelayMinutes                    float64
ArrDel15                           float64
Cancelled                

,Year,Quarter,Month,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,OriginState,...,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2015,1,1,4,2015-01-01,AA,N787AA,1,JFK,NY,...,390.0,402.0,378.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,5,2015-01-02,AA,N795AA,1,JFK,NY,...,390.0,381.0,357.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,6,2015-01-03,AA,N788AA,1,JFK,NY,...,390.0,358.0,330.0,2475.0,10,NaN,NaN,NaN,NaN,NaN


In [7]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
print(df["FlightDate"].dtype)
print(df["FlightDate"].min(), "to", df["FlightDate"].max())

datetime64[ns]
2015-01-01 00:00:00 to 2015-01-31 00:00:00


In [8]:
import gc

os.makedirs("../data/processed/ontime", exist_ok=True)

YEAR = 2015  # just this year for now

for month in range(1, 13):
    zip_filename = os.path.join(OUTPUT_DIR, f"ontime_{YEAR}_{month:02d}.zip")
    parquet_filename = f"../data/processed/ontime/ontime_{YEAR}_{month:02d}.parquet"

    if os.path.exists(parquet_filename):
        print(f"{YEAR}-{month:02d}: already processed, skipping")
        continue

    # Download if we don't already have the raw zip
    if not os.path.exists(zip_filename):
        url = BASE_URL.format(year=YEAR, month=month)
        print(f"{YEAR}-{month:02d}: downloading...", end=" ")
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            print(f"FAILED (status {r.status_code}) — skipping this month")
            continue
        with open(zip_filename, "wb") as f:
            f.write(r.content)
        print("done")
        time.sleep(1)

    # Extract, trim columns, fix dtype, save as parquet
    print(f"{YEAR}-{month:02d}: processing...", end=" ")
    with zipfile.ZipFile(zip_filename) as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]  # find the CSV regardless of exact name
        with z.open(csv_name) as f:
            month_df = pd.read_csv(f, usecols=KEEP_COLS)

    month_df["FlightDate"] = pd.to_datetime(month_df["FlightDate"])
    month_df.to_parquet(parquet_filename, index=False)

    print(f"saved {len(month_df):,} rows -> {parquet_filename}")

    del month_df
    gc.collect()  # free the memory before moving to the next month

print("\n2015 complete.")

2015-01: already processed, skipping
2015-02: already processed, skipping
2015-03: already processed, skipping
2015-04: already processed, skipping
2015-05: already processed, skipping
2015-06: already processed, skipping
2015-07: already processed, skipping
2015-08: already processed, skipping
2015-09: already processed, skipping
2015-10: already processed, skipping
2015-11: already processed, skipping
2015-12: already processed, skipping

2015 complete.


In [9]:
DTYPES = {"CancellationCode": "str"}

for YEAR in range(2016, 2026):
    for month in range(1, 13):
        zip_filename = os.path.join(OUTPUT_DIR, f"ontime_{YEAR}_{month:02d}.zip")
        parquet_filename = f"../data/processed/ontime/ontime_{YEAR}_{month:02d}.parquet"

        if os.path.exists(parquet_filename):
            print(f"{YEAR}-{month:02d}: already processed, skipping")
            continue

        if not os.path.exists(zip_filename):
            url = BASE_URL.format(year=YEAR, month=month)
            print(f"{YEAR}-{month:02d}: downloading...", end=" ")

            # Retry up to 3 times before giving up on this month
            for attempt in range(3):
                try:
                    r = requests.get(url, timeout=120)
                    if r.status_code == 200:
                        with open(zip_filename, "wb") as f:
                            f.write(r.content)
                        print("done")
                        break
                    else:
                        print(f"FAILED (status {r.status_code})")
                        break
                except requests.exceptions.RequestException as e:
                    print(f"attempt {attempt + 1} failed ({type(e).__name__}), retrying...", end=" ")
                    time.sleep(5)
            else:
                print(f"{YEAR}-{month:02d}: gave up after 3 attempts — skipping")
                continue

            time.sleep(1)

        print(f"{YEAR}-{month:02d}: processing...", end=" ")
        with zipfile.ZipFile(zip_filename) as z:
            csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(csv_name) as f:
                month_df = pd.read_csv(f, usecols=KEEP_COLS, dtype=DTYPES)

        month_df["FlightDate"] = pd.to_datetime(month_df["FlightDate"])
        month_df.to_parquet(parquet_filename, index=False)

        print(f"saved {len(month_df):,} rows -> {parquet_filename}")

        del month_df
        gc.collect()

print("\n2016-2025 complete.")

2016-01: already processed, skipping
2016-02: already processed, skipping
2016-03: already processed, skipping
2016-04: already processed, skipping
2016-05: already processed, skipping
2016-06: already processed, skipping
2016-07: already processed, skipping
2016-08: already processed, skipping
2016-09: already processed, skipping
2016-10: already processed, skipping
2016-11: already processed, skipping
2016-12: already processed, skipping
2017-01: already processed, skipping
2017-02: already processed, skipping
2017-03: already processed, skipping
2017-04: already processed, skipping
2017-05: already processed, skipping
2017-06: already processed, skipping
2017-07: already processed, skipping
2017-08: already processed, skipping
2017-09: already processed, skipping
2017-10: already processed, skipping
2017-11: already processed, skipping
2017-12: already processed, skipping
2018-01: already processed, skipping
2018-02: already processed, skipping
2018-03: already processed, skipping
2

In [10]:
import glob

parquet_files = sorted(glob.glob("../data/processed/ontime/*.parquet"))
print(f"Total files: {len(parquet_files)}")  # should be 132 (12 months x 11 years)

total_rows = 0
for f in parquet_files:
    total_rows += pd.read_parquet(f, columns=["Year"]).shape[0]

print(f"Total rows: {total_rows:,}")

Total files: 132
Total rows: 70,081,041


In [11]:
dupe_summary = []

for f in parquet_files:
    df = pd.read_parquet(f)

    # Check 1: exact duplicates — every single column identical, including delay times etc.
    # This would mean the same record got written twice, almost certainly a data error.
    exact_dupes = df.duplicated().sum()

    # Check 2: duplicates on the "natural key" — the columns that should uniquely
    # identify one specific flight. If these repeat but other columns (delay, etc.)
    # DIFFER, that's a more subtle problem: two conflicting records for what should
    # be one flight.
    key_cols = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest"]
    key_dupes = df.duplicated(subset=key_cols).sum()

    dupe_summary.append({
        "file": f.split("\\")[-1],  # backslash since you're on Windows
        "rows": len(df),
        "exact_dupes": exact_dupes,
        "key_dupes": key_dupes,
    })

    del df

summary_df = pd.DataFrame(dupe_summary)

print("Files with exact duplicate rows:")
print(summary_df[summary_df["exact_dupes"] > 0])

print("\nFiles with natural-key duplicates:")
print(summary_df[summary_df["key_dupes"] > 0])

print(f"\nTotal rows checked: {summary_df['rows'].sum():,}")
print(f"Total exact duplicate rows: {summary_df['exact_dupes'].sum():,}")
print(f"Total natural-key duplicate rows: {summary_df['key_dupes'].sum():,}")

Files with exact duplicate rows:
Empty DataFrame
Columns: [file, rows, exact_dupes, key_dupes]
Index: []

Files with natural-key duplicates:
                      file    rows  exact_dupes  key_dupes
29  ontime_2017_06.parquet  494266            0          1
41  ontime_2018_06.parquet  626216            0          1

Total rows checked: 70,081,041
Total exact duplicate rows: 0
Total natural-key duplicate rows: 2


In [12]:
key_cols = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest"]

files_to_check = ["ontime_2017_05.parquet", "ontime_2017_06.parquet", "ontime_2017_07.parquet",
                   "ontime_2017_11.parquet", "ontime_2018_06.parquet"]

for fname in files_to_check:
    df = pd.read_parquet(f"../data/processed/ontime/{fname}")
    dupe_mask = df.duplicated(subset=key_cols, keep=False)  # keep=False shows BOTH copies, not just the 2nd
    dupes = df[dupe_mask].sort_values(key_cols)
    print(f"\n--- {fname} ---")
    print(dupes[key_cols + ["Cancelled", "Diverted", "DepTime", "ArrTime", "ArrDelay"]])
    del df


--- ontime_2017_05.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []

--- ontime_2017_06.parquet ---
      FlightDate Reporting_Airline  Flight_Number_Reporting_Airline Origin  \
25874 2017-06-22                F9                             1740    SFO   
25914 2017-06-22                F9                             1740    SFO   

      Dest  Cancelled  Diverted  DepTime  ArrTime  ArrDelay  
25874  MCO        0.0       0.0    644.0   1213.0     199.0  
25914  MCO        0.0       0.0   2258.0    754.0      26.0  

--- ontime_2017_07.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []

--- ontime_2017_11.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, Cancelled, Diverted, 

In [13]:
key_cols_v2 = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"]

remaining_dupes = 0
for fname in files_to_check:
    df = pd.read_parquet(f"../data/processed/ontime/{fname}")
    count = df.duplicated(subset=key_cols_v2).sum()
    remaining_dupes += count
    if count > 0:
        print(f"{fname}: still {count} dupes even with CRSDepTime added")
    del df

print(f"\nTotal remaining dupes after adding CRSDepTime: {remaining_dupes}")


Total remaining dupes after adding CRSDepTime: 0


In [14]:
key_cols_v2 = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"]

for fname in ["ontime_2017_05.parquet", "ontime_2017_07.parquet", "ontime_2017_11.parquet", "ontime_2018_06.parquet"]:
    df = pd.read_parquet(f"../data/processed/ontime/{fname}")
    dupe_mask = df.duplicated(subset=key_cols_v2, keep=False)
    dupes = df[dupe_mask].sort_values(key_cols_v2)
    print(f"\n--- {fname} ---")
    print(dupes[key_cols_v2 + ["Cancelled", "Diverted", "DepTime", "ArrTime", "ArrDelay"]])
    del df


--- ontime_2017_05.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, CRSDepTime, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []

--- ontime_2017_07.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, CRSDepTime, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []

--- ontime_2017_11.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, CRSDepTime, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []

--- ontime_2018_06.parquet ---
Empty DataFrame
Columns: [FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, CRSDepTime, Cancelled, Diverted, DepTime, ArrTime, ArrDelay]
Index: []


In [15]:
def resolve_dupe(group):
    completed = group[group["ArrDelay"].notna()]
    if len(completed) == 1:
        # Exactly one row actually completed the flight — that's the real outcome
        return completed
    # Either both completed (conflicting reports, e.g. YX 3624) or neither did
    # (both cancelled, e.g. F9 170) — no principled way to prefer one row in
    # either case. Keep the first for reproducibility; documented as a known,
    # negligible-impact edge case (affects 4 rows out of 70M+).
    return group.iloc[[0]]

key_cols_v2 = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"]
affected_files = ["ontime_2017_05.parquet", "ontime_2017_07.parquet", "ontime_2017_11.parquet", "ontime_2018_06.parquet"]

for fname in affected_files:
    path = f"../data/processed/ontime/{fname}"
    df = pd.read_parquet(path)

    dupe_mask = df.duplicated(subset=key_cols_v2, keep=False)
    clean_part = df[~dupe_mask]
    dupe_part = df[dupe_mask]

    resolved = dupe_part.groupby(key_cols_v2, group_keys=False).apply(resolve_dupe)
    final_df = pd.concat([clean_part, resolved], ignore_index=True)

    print(f"{fname}: {len(df)} -> {len(final_df)} rows ({len(df) - len(final_df)} dropped)")
    final_df.to_parquet(path, index=False)  # overwrite with cleaned version
    del df, final_df

C:\Users\Ayush Chaudhary\AppData\Local\Temp\ipykernel_6440\1312608074.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resolved = dupe_part.groupby(key_cols_v2, group_keys=False).apply(resolve_dupe)


ontime_2017_05.parquet: 486482 -> 486482 rows (0 dropped)


C:\Users\Ayush Chaudhary\AppData\Local\Temp\ipykernel_6440\1312608074.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resolved = dupe_part.groupby(key_cols_v2, group_keys=False).apply(resolve_dupe)


ontime_2017_07.parquet: 509069 -> 509069 rows (0 dropped)


C:\Users\Ayush Chaudhary\AppData\Local\Temp\ipykernel_6440\1312608074.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resolved = dupe_part.groupby(key_cols_v2, group_keys=False).apply(resolve_dupe)


ontime_2017_11.parquet: 454161 -> 454161 rows (0 dropped)


C:\Users\Ayush Chaudhary\AppData\Local\Temp\ipykernel_6440\1312608074.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resolved = dupe_part.groupby(key_cols_v2, group_keys=False).apply(resolve_dupe)


ontime_2018_06.parquet: 626216 -> 626216 rows (0 dropped)


In [16]:
key_cols_v2 = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"]

total_dupes = 0
total_rows = 0

for f in parquet_files:
    df = pd.read_parquet(f)
    total_dupes += df.duplicated(subset=key_cols_v2).sum()
    total_rows += len(df)
    del df

print(f"Total rows: {total_rows:,}")
print(f"Remaining duplicates: {total_dupes}")

Total rows: 70,081,041
Remaining duplicates: 0


In [17]:
DB1B_BASE_URL = "https://transtats.bts.gov/PREZIP/Origin_and_Destination_Survey_DB1BMarket_{year}_{quarter}.zip"
DB1B_OUTPUT_DIR = "../data/raw/db1b"
os.makedirs(DB1B_OUTPUT_DIR, exist_ok=True)

year, quarter = 2015, 1  # just one quarter for now

filename = os.path.join(DB1B_OUTPUT_DIR, f"db1b_market_{year}_q{quarter}.zip")
url = DB1B_BASE_URL.format(year=year, quarter=quarter)

print(f"Downloading {year} Q{quarter} from {url} ...")
r = requests.get(url, timeout=120)
print(f"Status code: {r.status_code}")

if r.status_code == 200:
    with open(filename, "wb") as f:
        f.write(r.content)
    print(f"Saved {filename} ({len(r.content) / 1_000_000:.1f} MB)")
else:
    print("Didn't work — paste me the status code and I'll help debug.")

Status code: 200
Saved ../data/raw/db1b\db1b_market_2015_q1.zip (82.9 MB)


In [18]:
with zipfile.ZipFile(filename) as z:
    print("Files inside the zip:")
    for name in z.namelist():
        print(" -", name)

Files inside the zip:
 - Origin_and_Destination_Survey_DB1BMarket_2015_1.csv
 - readme.html


In [19]:
csv_name = "Origin_and_Destination_Survey_DB1BMarket_2015_1.csv"

with zipfile.ZipFile(filename) as z:
    with z.open(csv_name) as f:
        db1b_df = pd.read_csv(f, nrows=1000)

print(f"Shape (first 1000 rows only): {db1b_df.shape}")
print(f"\nTotal columns: {len(db1b_df.columns)}")
print("\nColumn names:")
for col in db1b_df.columns:
    print(" -", col)

Shape (first 1000 rows only): (1000, 42)

Total columns: 42

Column names:
 - ItinID
 - MktID
 - MktCoupons
 - Year
 - Quarter
 - OriginAirportID
 - OriginAirportSeqID
 - OriginCityMarketID
 - Origin
 - OriginCountry
 - OriginStateFips
 - OriginState
 - OriginStateName
 - OriginWac
 - DestAirportID
 - DestAirportSeqID
 - DestCityMarketID
 - Dest
 - DestCountry
 - DestStateFips
 - DestState
 - DestStateName
 - DestWac
 - AirportGroup
 - WacGroup
 - TkCarrierChange
 - TkCarrierGroup
 - OpCarrierChange
 - OpCarrierGroup
 - RPCarrier
 - TkCarrier
 - OpCarrier
 - BulkFare
 - Passengers
 - MktFare
 - MktDistance
 - MktDistanceGroup
 - MktMilesFlown
 - NonStopMiles
 - ItinGeoType
 - MktGeoType
 - Unnamed: 41


In [20]:
KEEP_COLS_DB1B = [
    "Year", "Quarter",
    "Origin", "OriginState", "Dest", "DestState",
    "OpCarrier", "RPCarrier",  # keep both — OpCarrier for the join, RPCarrier in case we need it later
    "Passengers", "MktFare", "MktDistance", "MktDistanceGroup", "MktMilesFlown",
    "BulkFare", "MktGeoType",  # needed to apply the filtering rule, not for the join itself
]

In [21]:
with zipfile.ZipFile(filename) as z:
    with z.open(csv_name) as f:
        db1b_df = pd.read_csv(f, usecols=KEEP_COLS_DB1B)

print(f"Shape: {db1b_df.shape}")
print(f"\nDtypes:\n{db1b_df.dtypes}")
print(f"\nFare distribution:\n{db1b_df['MktFare'].describe()}")
print(f"\nBulkFare value counts:\n{db1b_df['BulkFare'].value_counts()}")
print(f"\nRows with MktFare under $25: {(db1b_df['MktFare'] < 25).sum():,} out of {len(db1b_df):,}")

C:\Users\Ayush Chaudhary\AppData\Local\Temp\ipykernel_6440\3892133575.py:3: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  db1b_df = pd.read_csv(f, usecols=KEEP_COLS_DB1B)


Shape: (5582629, 15)

Dtypes:
Year                  int64
Quarter               int64
Origin               object
OriginState          object
Dest                 object
DestState            object
RPCarrier            object
OpCarrier            object
BulkFare            float64
Passengers          float64
MktFare             float64
MktDistance         float64
MktDistanceGroup      int64
MktMilesFlown       float64
MktGeoType            int64
dtype: object

Fare distribution:
count    5.582629e+06
mean     2.645916e+02
std      2.961858e+02
min      0.000000e+00
25%      1.550000e+02
50%      2.210400e+02
75%      3.245000e+02
max      2.199500e+05
Name: MktFare, dtype: float64

BulkFare value counts:
BulkFare
0.0    5581650
1.0        979
Name: count, dtype: int64

Rows with MktFare under $25: 280,293 out of 5,582,629


In [22]:
total_passengers = db1b_df["Passengers"].sum()
low_fare_passengers = db1b_df.loc[db1b_df["MktFare"] < 25, "Passengers"].sum()

print(f"Total passengers: {total_passengers:,.0f}")
print(f"Passengers on sub-$25 fares: {low_fare_passengers:,.0f} ({100 * low_fare_passengers / total_passengers:.2f}%)")

print("\n--- Sample of sub-$25 fares ---")
print(db1b_df[db1b_df["MktFare"] < 25].sort_values("Passengers", ascending=False)
      [["Origin", "Dest", "OpCarrier", "MktFare", "Passengers", "MktDistance"]].head(10))

print("\n--- Sample of highest fares ---")
print(db1b_df.sort_values("MktFare", ascending=False)
      [["Origin", "Dest", "OpCarrier", "MktFare", "Passengers", "MktDistance"]].head(10))

Total passengers: 10,794,748
Passengers on sub-$25 fares: 840,566 (7.79%)

--- Sample of sub-$25 fares ---
        Origin Dest OpCarrier  MktFare  Passengers  MktDistance
4195898    FLL  BWI        WN      5.5       719.0        925.0
4195897    BWI  FLL        WN      5.5       719.0        925.0
4202509    BWI  MCO        WN      5.5       558.0        787.0
4202510    MCO  BWI        WN      5.5       558.0        787.0
2138939    MCO  MDW        WN      5.5       557.0        990.0
2138938    MDW  MCO        WN      5.5       557.0        990.0
2144414    PHX  MDW        WN      5.5       552.0       1444.0
2144413    MDW  PHX        WN      5.5       552.0       1444.0
2147013    RSW  MDW        WN      5.5       537.0       1105.0
2147012    MDW  RSW        WN      5.5       537.0       1105.0

--- Sample of highest fares ---
        Origin Dest OpCarrier   MktFare  Passengers  MktDistance
3482197    INL  MSP        OO  219950.0         1.0        255.0
3482196    MSP  INL       

In [23]:
# How pervasive is the exact $5.50 WN pattern?
wn_550 = db1b_df[(db1b_df["OpCarrier"] == "WN") & (db1b_df["MktFare"] == 5.5)]
print(f"Exact $5.50 WN rows: {len(wn_550):,}, representing {wn_550['Passengers'].sum():,.0f} passengers")

# Fare per mile — a much more stable metric than raw fare, since it normalizes for distance
db1b_df["fare_per_mile"] = db1b_df["MktFare"] / db1b_df["MktMilesFlown"].replace(0, pd.NA)
print(f"\nFare-per-mile distribution:\n{db1b_df['fare_per_mile'].describe()}")

# The MSP-INL cluster specifically
print(f"\n--- MSP-INL rows ---")
print(db1b_df[((db1b_df["Origin"] == "MSP") & (db1b_df["Dest"] == "INL")) |
              ((db1b_df["Origin"] == "INL") & (db1b_df["Dest"] == "MSP"))]
      [["Origin", "Dest", "OpCarrier", "MktFare", "Passengers", "MktMilesFlown"]])

# 99.9th percentile fare-per-mile as a sanity check on how extreme the top end really is
print(f"\n99th / 99.9th / 99.99th percentile fare-per-mile:")
print(db1b_df["fare_per_mile"].quantile([0.99, 0.999, 0.9999]))

Exact $5.50 WN rows: 14,512, representing 167,185 passengers

Fare-per-mile distribution:
count     5581045.0
unique    2123679.0
top             0.0
freq        64091.0
Name: fare_per_mile, dtype: float64

--- MSP-INL rows ---
        Origin Dest OpCarrier    MktFare  Passengers  MktMilesFlown
2744579    INL  MSP        OO     118.09         1.0          255.0
2872152    MSP  INL        OO     188.40         1.0          255.0
3441037    INL  MSP        OO     206.00         1.0          255.0
3441038    INL  MSP        OO     332.00         1.0          255.0
3441095    INL  MSP        OO      56.94         1.0          255.0
3441096    MSP  INL        OO      58.06         1.0          260.0
3441099    INL  MSP        OO       5.50         1.0          255.0
3441100    MSP  INL        OO       5.50         1.0          255.0
3441101    INL  MSP        OO     142.00         1.0          255.0
3441102    MSP  INL        DL     142.00         1.0          255.0
3441103    INL  MSP     

In [24]:
# 1. Outlier filter — data-driven, not an arbitrary guess.
# Anything beyond ~10x the legitimate 99.99th percentile is not a real fare.
outlier_threshold = db1b_df["fare_per_mile"].quantile(0.9999) * 10
outliers = db1b_df[db1b_df["fare_per_mile"] > outlier_threshold]
print(f"Outlier threshold: ${outlier_threshold:.2f}/mile")
print(f"Rows flagged as data errors: {len(outliers)}, representing {outliers['Passengers'].sum():.0f} passengers")
print(outliers[["Origin", "Dest", "OpCarrier", "MktFare", "Passengers", "fare_per_mile"]])

Outlier threshold: $89.28/mile
Rows flagged as data errors: 11, representing 13 passengers
        Origin Dest OpCarrier   MktFare  Passengers fare_per_mile
1023529    ORD  MSN        EV    9999.0         2.0     92.583333
3349980    BTM  SLC        OO   99130.0         2.0    276.899441
3358731    CLD  LAX        OO    9999.0         1.0    116.267442
3479924    MSP  ABR        OO   29460.0         1.0     114.63035
3479925    MSP  ABR        OO   30780.0         1.0    119.766537
3482194    MSP  INL        OO  207000.0         1.0    811.764706
3482195    INL  MSP        DL  207000.0         1.0    811.764706
3482196    MSP  INL        OO  219950.0         1.0     862.54902
3482197    INL  MSP        OO  219950.0         1.0     862.54902
3505102    ORD  MKE        OO    9999.0         1.0    149.238806
4606710    LAX  LAS        AA   69345.0         1.0    293.834746


In [25]:
# Data-quality fix #1: remove true fare data errors (not real fares — includes
# sentinel/placeholder values like exactly $9999, and fares 10x+ beyond the
# legitimate 99.99th percentile fare-per-mile). Confirmed: 11 rows / 13 passengers
# out of 5.58M rows in this quarter alone — negligible volume, clearly erroneous.
outlier_threshold = db1b_df["fare_per_mile"].quantile(0.9999) * 10
db1b_clean = db1b_df[db1b_df["fare_per_mile"] <= outlier_threshold].copy()

# Data-quality fix #2: remove the near-zero, non-BulkFare-flagged records
# (BulkFare==1 is already a separate, tiny group — filter that too while we're here).
db1b_clean = db1b_clean[(db1b_clean["MktFare"] >= 25) | (db1b_clean["MktFare"].isna())]
db1b_clean = db1b_clean[db1b_clean["BulkFare"] != 1]

print(f"Original: {len(db1b_df):,} rows")
print(f"After cleaning: {len(db1b_clean):,} rows ({len(db1b_df) - len(db1b_clean):,} dropped)")
print(f"\nNote: WN's known $5.50-cluster proration quirk is NOT filtered — it's real")
print(f"data reflecting a documented DB1B methodology limitation for Southwest,")
print(f"not an error. Will be called out explicitly in the README/fare analysis.")

Original: 5,582,629 rows
After cleaning: 5,302,325 rows (280,304 dropped)

Note: WN's known $5.50-cluster proration quirk is NOT filtered — it's real
data reflecting a documented DB1B methodology limitation for Southwest,
not an error. Will be called out explicitly in the README/fare analysis.


In [26]:
DB1B_KEEP_COLS = [
    "Year", "Quarter",
    "Origin", "OriginState", "Dest", "DestState",
    "OpCarrier", "RPCarrier",
    "Passengers", "MktFare", "MktDistance", "MktDistanceGroup", "MktMilesFlown",
    "BulkFare", "MktGeoType",
]

DB1B_DTYPES = {
    "Origin": "str", "OriginState": "str",
    "Dest": "str", "DestState": "str",
    "OpCarrier": "str", "RPCarrier": "str",
}

os.makedirs("../data/processed/db1b", exist_ok=True)

for year in range(2015, 2026):
    max_quarter = 2 if year == 2025 else 4  # 2025 only has Q1-Q2 before the DB1B cutover

    for quarter in range(1, max_quarter + 1):
        zip_filename = os.path.join(DB1B_OUTPUT_DIR, f"db1b_market_{year}_q{quarter}.zip")
        parquet_filename = f"../data/processed/db1b/db1b_{year}_q{quarter}.parquet"

        if os.path.exists(parquet_filename):
            print(f"{year} Q{quarter}: already processed, skipping")
            continue

        if not os.path.exists(zip_filename):
            url = DB1B_BASE_URL.format(year=year, quarter=quarter)
            print(f"{year} Q{quarter}: downloading...", end=" ")

            for attempt in range(3):
                try:
                    r = requests.get(url, timeout=180)
                    if r.status_code == 200:
                        with open(zip_filename, "wb") as f:
                            f.write(r.content)
                        print("done")
                        break
                    else:
                        print(f"FAILED (status {r.status_code})")
                        break
                except requests.exceptions.RequestException as e:
                    print(f"attempt {attempt + 1} failed ({type(e).__name__}), retrying...", end=" ")
                    time.sleep(5)
            else:
                print(f"{year} Q{quarter}: gave up after 3 attempts — skipping")
                continue

            time.sleep(1)

        print(f"{year} Q{quarter}: processing...", end=" ")
        with zipfile.ZipFile(zip_filename) as z:
            csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(csv_name) as f:
                q_df = pd.read_csv(f, usecols=DB1B_KEEP_COLS, dtype=DB1B_DTYPES)

        q_df["fare_per_mile"] = q_df["MktFare"] / q_df["MktMilesFlown"].replace(0, pd.NA)
        threshold = q_df["fare_per_mile"].quantile(0.9999) * 10
        q_df = q_df[q_df["fare_per_mile"] <= threshold]
        q_df = q_df[(q_df["MktFare"] >= 25) | (q_df["MktFare"].isna())]
        q_df = q_df[q_df["BulkFare"] != 1]
        q_df = q_df.drop(columns=["fare_per_mile"])

        q_df.to_parquet(parquet_filename, index=False)
        print(f"saved {len(q_df):,} rows -> {parquet_filename}")

        del q_df
        gc.collect()

print("\nDB1B (2015 Q1 - 2025 Q2) complete.")

2015 Q1: already processed, skipping
2015 Q2: already processed, skipping
2015 Q3: already processed, skipping
2015 Q4: already processed, skipping
2016 Q1: already processed, skipping
2016 Q2: already processed, skipping
2016 Q3: already processed, skipping
2016 Q4: already processed, skipping
2017 Q1: already processed, skipping
2017 Q2: already processed, skipping
2017 Q3: already processed, skipping
2017 Q4: already processed, skipping
2018 Q1: already processed, skipping
2018 Q2: already processed, skipping
2018 Q3: already processed, skipping
2018 Q4: already processed, skipping
2019 Q1: already processed, skipping
2019 Q2: already processed, skipping
2019 Q3: already processed, skipping
2019 Q4: already processed, skipping
2020 Q1: already processed, skipping
2020 Q2: already processed, skipping
2020 Q3: already processed, skipping
2020 Q4: already processed, skipping
2021 Q1: already processed, skipping
2021 Q2: already processed, skipping
2021 Q3: already processed, skipping
2

In [27]:
import glob

db1b_files = sorted(glob.glob("../data/processed/db1b/*.parquet"))
print(f"Total files: {len(db1b_files)}")  # should be 42 (4 quarters x 10 years, + 2 for 2025)

total_rows = 0
for f in db1b_files:
    total_rows += pd.read_parquet(f, columns=["Year"]).shape[0]

print(f"Total rows: {total_rows:,}")

# Check exactly which quarters we have, to catch any silent gaps
expected = [f"db1b_{y}_q{q}.parquet" for y in range(2015, 2025) for q in range(1, 5)] + \
           [f"db1b_2025_q{q}.parquet" for q in range(1, 3)]
actual = [f.split("\\")[-1].split("/")[-1] for f in db1b_files]  # handles both Windows and Unix path separators
missing = set(expected) - set(actual)

if missing:
    print(f"\nMISSING quarters: {sorted(missing)}")
else:
    print("\nAll 42 expected quarters present.")

Total files: 42
Total rows: 257,173,898

All 42 expected quarters present.


In [28]:
dupe_summary = []

for f in db1b_files:
    df = pd.read_parquet(f)
    exact_dupes = df.duplicated().sum()
    dupe_summary.append({
        "file": f.split("\\")[-1].split("/")[-1],
        "rows": len(df),
        "exact_dupes": exact_dupes,
    })
    del df

summary_df = pd.DataFrame(dupe_summary)
print("Files with exact duplicate rows:")
print(summary_df[summary_df["exact_dupes"] > 0])
print(f"\nTotal exact duplicate rows: {summary_df['exact_dupes'].sum():,}")

Files with exact duplicate rows:
                    file     rows  exact_dupes
0   db1b_2015_q1.parquet  5302325       314200
1   db1b_2015_q2.parquet  6271111       421065
2   db1b_2015_q3.parquet  6093394       356782
3   db1b_2015_q4.parquet  6071630       375361
4   db1b_2016_q1.parquet  5554846       323865
5   db1b_2016_q2.parquet  6264791       380915
6   db1b_2016_q3.parquet  6130422       365506
7   db1b_2016_q4.parquet  6201278       394221
8   db1b_2017_q1.parquet  5529283       317916
9   db1b_2017_q2.parquet  6210403       366407
10  db1b_2017_q3.parquet  5931944       345900
11  db1b_2017_q4.parquet  6455073       406541
12  db1b_2018_q1.parquet  5729138       335535
13  db1b_2018_q2.parquet  6740067       414749
14  db1b_2018_q3.parquet  6375040       378711
15  db1b_2018_q4.parquet  6733032       424939
16  db1b_2019_q1.parquet  6121288       353779
17  db1b_2019_q2.parquet  7041715       436091
18  db1b_2019_q3.parquet  6718566       388516
19  db1b_2019_q4.parquet  6

In [29]:
# Reload Q1 2015 WITH the ID columns this time, to see if "duplicate" rows
# actually have different ItinIDs (proving they're separate real tickets)
with zipfile.ZipFile(os.path.join(DB1B_OUTPUT_DIR, "db1b_market_2015_q1.zip")) as z:
    with z.open("Origin_and_Destination_Survey_DB1BMarket_2015_1.csv") as f:
        check_df = pd.read_csv(f, usecols=DB1B_KEEP_COLS + ["ItinID", "MktID"], dtype=DB1B_DTYPES)

# Find rows that are duplicates when ignoring ID columns
no_id_cols = [c for c in DB1B_KEEP_COLS]
dupe_mask = check_df.duplicated(subset=no_id_cols, keep=False)
sample_dupes = check_df[dupe_mask].sort_values(no_id_cols).head(10)

print(sample_dupes[["ItinID", "MktID", "Origin", "Dest", "OpCarrier", "MktFare", "Passengers"]])

               ItinID           MktID Origin Dest OpCarrier  MktFare  \
433055    20151967392   2015196739201    ABE  ABQ        DL   482.00   
433858    20151967917   2015196791703    ABE  ABQ        DL   482.00   
3854182  201513145517  20151314551701    ABE  ATL        99   214.58   
3860198  201513150200  20151315020003    ABE  ATL        99   214.58   
432949    20151967339   2015196733901    ABE  ATL        DL     5.50   
432951    20151967340   2015196734001    ABE  ATL        DL     5.50   
432991    20151967360   2015196736001    ABE  ATL        DL   168.00   
432993    20151967361   2015196736101    ABE  ATL        DL   168.00   
432947    20151967337   2015196733701    ABE  ATL        DL   497.00   
433037    20151967383   2015196738301    ABE  ATL        DL   497.00   

         Passengers  
433055          1.0  
433858          1.0  
3854182         1.0  
3860198         1.0  
432949          1.0  
432951          1.0  
432991          1.0  
432993          1.0  
432947   

In [30]:
DB1C_URLS = {
    "2025_07": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202507.22MAY2026.zip",
    "2025_08": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202508.22MAY2026.zip",
    "2025_09": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202509.22MAY2026.zip",
    "2025_10": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202510.22MAY2026.zip",
    "2025_11": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202511.26MAY2026.zip",
    "2025_12": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202512.27MAY2026.zip",
    "2026_01": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202601.01JUN2026.zip",
    "2026_02": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202602.02JUN2026.zip",
    "2026_03": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202603.02JUL2026.zip",
    "2026_04": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202604.15JUL2026.zip",
    "2026_05": "https://ostrapeispubdownloadprod.blob.core.windows.net/ostrapeis-pub-download-prod/ond40/products/db1c_public/DB1C.MARKET.202605.20AUG2026.zip",
}

DB1C_OUTPUT_DIR = "../data/raw/db1c"
os.makedirs(DB1C_OUTPUT_DIR, exist_ok=True)

for period, url in DB1C_URLS.items():
    filename = os.path.join(DB1C_OUTPUT_DIR, f"db1c_{period}.zip")

    if os.path.exists(filename):
        print(f"{period}: already downloaded, skipping")
        continue

    print(f"{period}: downloading...", end=" ")
    try:
        r = requests.get(url, timeout=180)
        if r.status_code == 200:
            with open(filename, "wb") as f:
                f.write(r.content)
            print(f"done ({len(r.content) / 1_000_000:.1f} MB)")
        else:
            print(f"FAILED (status {r.status_code})")
    except requests.exceptions.RequestException as e:
        print(f"error: {e}")

    time.sleep(1)

print("\nDB1C download complete.")

2025_07: already downloaded, skipping
2025_08: already downloaded, skipping
2025_09: already downloaded, skipping
2025_10: already downloaded, skipping
2025_11: already downloaded, skipping
2025_12: already downloaded, skipping
2026_01: already downloaded, skipping
2026_02: already downloaded, skipping
2026_03: already downloaded, skipping
2026_04: already downloaded, skipping
2026_05: already downloaded, skipping

DB1C download complete.


In [31]:
sample_zip = os.path.join(DB1C_OUTPUT_DIR, "db1c_2025_07.zip")

with zipfile.ZipFile(sample_zip) as z:
    print("Files inside the zip:")
    for name in z.namelist():
        print(" -", name)

Files inside the zip:
 - DB1C.MARKET.202507.22MAY2026.asc.zip
 - DB1C.MARKET.202507.22MAY2026.csv.zip
 - DB1C.MARKET.202507.22MAY2026.parquet


In [32]:
with zipfile.ZipFile(sample_zip) as z:
    z.extract("DB1C.MARKET.202507.22MAY2026.parquet", DB1C_OUTPUT_DIR)

db1c_sample = pd.read_parquet(os.path.join(DB1C_OUTPUT_DIR, "DB1C.MARKET.202507.22MAY2026.parquet"))

print(f"Shape: {db1c_sample.shape}")
print(f"\nColumn names:")
for col in db1c_sample.columns:
    print(" -", col)

Shape: (23106046, 57)

Column names:
 - ItinID
 - MktID
 - GateID
 - MktCoupons
 - RpYear
 - RpQuarter
 - RpMonth
 - SchFlYear
 - SchFlQuarter
 - SchFlMonth
 - OriginAirportID
 - OriginAirportSeqID
 - OriginCityMarketID
 - Origin
 - OriginCountry
 - OriginStateFips
 - OriginState
 - OriginStateName
 - OriginWac
 - DestAirportID
 - DestAirportSeqID
 - DestCityMarketID
 - Dest
 - DestCountry
 - DestStateFips
 - DestState
 - DestStateName
 - DestWac
 - AirportGroup
 - WacGroup
 - DwellTimeGroup
 - SegmentViaGroup
 - RpCarrierAirlineID
 - RpCarrier
 - IssuingCarrierAirlineID
 - IssuingCarrier
 - MktCarrierAirlineID
 - MktCarrier
 - MktCarrierGroup
 - MktCarrierChange
 - OpCarrierAirlineID
 - OpCarrier
 - OpCarrierGroup
 - OpCarrierChange
 - Passengers
 - MktAmount
 - MktTax
 - V_Yield
 - PurchaseWindowGroup
 - TotalDistance
 - MilesTraveled
 - NonStopMiles
 - MktDistanceGroup
 - Nonstop
 - MktGeoType
 - ItinGeoType
 - Break_LegacyLogic


In [33]:
# 1. Compare average fare magnitude: does MktAmount alone, or MktAmount minus tax,
# land closer to what we saw in DB1B (2015 Q1 mean MktFare was ~$264.59)?
print("MktAmount stats:")
print(db1c_sample["MktAmount"].describe())

print("\nMktTax stats:")
print(db1c_sample["MktTax"].describe())

db1c_sample["fare_excl_tax"] = db1c_sample["MktAmount"] - db1c_sample["MktTax"]
print("\nMktAmount minus MktTax stats:")
print(db1c_sample["fare_excl_tax"].describe())

# 2. Compare the three distance fields against each other and against MktDistanceGroup
# to see which one the grouping is actually derived from
print("\nDistance field comparison (first 10 rows):")
print(db1c_sample[["TotalDistance", "MilesTraveled", "NonStopMiles", "MktDistanceGroup"]].head(10))

print("\nCorrelation between distance fields:")
print(db1c_sample[["TotalDistance", "MilesTraveled", "NonStopMiles"]].corr())

MktAmount stats:
count    2.310543e+07
mean     2.119193e+02
std      1.785329e+02
min      0.000000e+00
25%      1.032100e+02
50%      1.838100e+02
75%      2.809800e+02
max      1.780976e+04
Name: MktAmount, dtype: float64

MktTax stats:
count    2.310543e+07
mean     2.882917e+01
std      1.482933e+01
min      0.000000e+00
25%      2.042000e+01
50%      2.785000e+01
75%      3.658000e+01
max      1.932000e+03
Name: MktTax, dtype: float64

MktAmount minus MktTax stats:
count    2.310543e+07
mean     1.830901e+02
std      1.657048e+02
min      0.000000e+00
25%      8.244000e+01
50%      1.551500e+02
75%      2.452900e+02
max      1.761495e+04
Name: fare_excl_tax, dtype: float64

Distance field comparison (first 10 rows):
   TotalDistance  MilesTraveled  NonStopMiles  MktDistanceGroup
0          121.0          121.0         121.0               1.0
1          121.0          121.0         121.0               1.0
2          121.0          121.0         121.0               1.0
3          1

In [34]:
DB1C_COLUMN_MAP = {
    "SchFlYear": "Year",
    "SchFlQuarter": "Quarter",
    "Origin": "Origin", "OriginState": "OriginState",
    "Dest": "Dest", "DestState": "DestState",
    "OpCarrier": "OpCarrier", "RpCarrier": "RPCarrier",
    "Passengers": "Passengers",
    "NonStopMiles": "MktDistance",
    "MktDistanceGroup": "MktDistanceGroup",
    "MilesTraveled": "MktMilesFlown",
    "MktGeoType": "MktGeoType",
}

DB1C_KEEP_RAW = list(DB1C_COLUMN_MAP.keys()) + ["MktAmount", "MktTax"]

os.makedirs("../data/processed/db1c", exist_ok=True)

for period in DB1C_URLS.keys():
    zip_path = os.path.join(DB1C_OUTPUT_DIR, f"db1c_{period}.zip")
    parquet_out = f"../data/processed/db1c/db1c_{period}.parquet"

    if os.path.exists(parquet_out):
        print(f"{period}: already processed, skipping")
        continue

    with zipfile.ZipFile(zip_path) as z:
        pq_name = [n for n in z.namelist() if n.endswith(".parquet")][0]
        z.extract(pq_name, DB1C_OUTPUT_DIR)
        m_df = pd.read_parquet(os.path.join(DB1C_OUTPUT_DIR, pq_name), columns=DB1C_KEEP_RAW)

    m_df["MktFare"] = m_df["MktAmount"] - m_df["MktTax"]
    m_df = m_df.rename(columns=DB1C_COLUMN_MAP)
    m_df["BulkFare"] = pd.NA  # no equivalent flag exists in DB1C — documented gap, not filtered

    # Same outlier + floor cleaning rules as DB1B, recalculated per period
    m_df["fare_per_mile"] = m_df["MktFare"] / m_df["MktMilesFlown"].replace(0, pd.NA)
    threshold = m_df["fare_per_mile"].quantile(0.9999) * 10
    m_df = m_df[m_df["fare_per_mile"] <= threshold]
    m_df = m_df[(m_df["MktFare"] >= 25) | (m_df["MktFare"].isna())]
    m_df = m_df.drop(columns=["fare_per_mile", "MktAmount", "MktTax"])

    m_df.to_parquet(parquet_out, index=False)
    print(f"{period}: saved {len(m_df):,} rows -> {parquet_out}")

    del m_df
    gc.collect()

print("\nDB1C processing complete.")

2025_07: already processed, skipping
2025_08: already processed, skipping
2025_09: already processed, skipping
2025_10: already processed, skipping
2025_11: already processed, skipping
2025_12: already processed, skipping
2026_01: already processed, skipping
2026_02: already processed, skipping
2026_03: already processed, skipping
2026_04: already processed, skipping
2026_05: already processed, skipping

DB1C processing complete.


In [35]:
db1c_files = sorted(glob.glob("../data/processed/db1c/*.parquet"))
print(f"Total DB1C files: {len(db1c_files)}")  # should be 11

total_db1c_rows = 0
for f in db1c_files:
    total_db1c_rows += pd.read_parquet(f, columns=["Year"]).shape[0]

print(f"Total DB1C rows: {total_db1c_rows:,}")
print(f"Total DB1B rows (from earlier): 257,173,898")
print(f"Combined fare dataset: {total_db1c_rows + 257173898:,} rows, covering 2015 Q1 - 2026 May")

Total DB1C files: 11
Total DB1C rows: 196,753,373
Total DB1B rows (from earlier): 257,173,898
Combined fare dataset: 453,927,271 rows, covering 2015 Q1 - 2026 May


In [36]:
import os

def dir_stats(path):
    files = [f for f in os.listdir(path) if f.endswith(".parquet")]
    sizes = [os.path.getsize(os.path.join(path, f)) / 1_000_000 for f in files]
    return len(files), sum(sizes), max(sizes) if sizes else 0

for label, path in [("on-time", "../data/processed/ontime"),
                     ("db1b", "../data/processed/db1b"),
                     ("db1c", "../data/processed/db1c")]:
    count, total_mb, largest_mb = dir_stats(path)
    print(f"{label}: {count} files, {total_mb:,.0f} MB total, largest file {largest_mb:,.0f} MB")

on-time: 132 files, 1,408 MB total, largest file 14 MB
db1b: 42 files, 1,298 MB total, largest file 38 MB
db1c: 11 files, 1,332 MB total, largest file 194 MB


In [37]:
import glob

# Distinct carrier codes across all on-time files
carrier_codes = set()
for f in glob.glob("../data/processed/ontime/*.parquet"):
    codes = pd.read_parquet(f, columns=["Reporting_Airline"])["Reporting_Airline"].unique()
    carrier_codes.update(codes)

print(f"Distinct carrier codes: {len(carrier_codes)}")
print(sorted(carrier_codes))

# Distinct airport codes (both origin and destination)
airport_codes = set()
for f in glob.glob("../data/processed/ontime/*.parquet"):
    df = pd.read_parquet(f, columns=["Origin", "Dest"])
    airport_codes.update(df["Origin"].unique())
    airport_codes.update(df["Dest"].unique())

print(f"\nDistinct airport codes: {len(airport_codes)}")

Distinct carrier codes: 20
['9E', 'AA', 'AS', 'B6', 'DL', 'EV', 'F9', 'G4', 'HA', 'MQ', 'NK', 'OH', 'OO', 'QX', 'UA', 'US', 'VX', 'WN', 'YV', 'YX']

Distinct airport codes: 395


In [38]:
CARRIER_NAMES = {
    "9E": "Endeavor Air", "AA": "American Airlines", "AS": "Alaska Airlines",
    "B6": "JetBlue Airways", "DL": "Delta Air Lines", "EV": "ExpressJet Airlines",
    "F9": "Frontier Airlines", "G4": "Allegiant Air", "HA": "Hawaiian Airlines",
    "MQ": "Envoy Air", "NK": "Spirit Airlines", "OH": "PSA Airlines",
    "OO": "SkyWest Airlines", "QX": "Horizon Air", "UA": "United Airlines",
    "US": "US Airways", "VX": "Virgin America", "WN": "Southwest Airlines",
    "YV": "Mesa Airlines", "YX": "Republic Airway",
}

In [39]:
airports_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
airports_raw = pd.read_csv(
    airports_url,
    header=None,
    names=["airport_id", "name", "city", "country", "iata", "icao",
           "lat", "lon", "altitude", "timezone", "dst", "tz_db", "type", "source"]
)

# Filter to just the airports actually in our data, US only
airports_lookup = airports_raw[
    (airports_raw["iata"].isin(airport_codes)) & (airports_raw["country"] == "United States")
][["iata", "name", "city"]]

print(f"Matched: {len(airports_lookup)} out of {len(airport_codes)} airport codes")
missing = airport_codes - set(airports_lookup["iata"])
print(f"Unmatched codes: {sorted(missing)}")

Matched: 383 out of 395 airport codes
Unmatched codes: ['BQN', 'EAR', 'GUM', 'IFP', 'PPG', 'PSE', 'SJU', 'SPN', 'STT', 'STX', 'TKI', 'XWA']


In [40]:
# Re-check the unmatched codes without the country filter, to see what country they're actually listed under
check = airports_raw[airports_raw["iata"].isin(missing)][["iata", "name", "city", "country"]]
print(check)

     iata                                      name              city  \
1902  PPG           Pago Pago International Airport         Pago Pago   
2149  SPN              Saipan International Airport            Saipan   
2151  GUM  Antonio B. Won Pat International Airport             Agana   
2738  STT                     Cyril E. King Airport        St. Thomas   
2739  STX                   Henry E Rohlsen Airport  St. Croix Island   
2740  BQN                  Rafael Hernandez Airport         Aguadilla   
2744  PSE                         Mercedita Airport             Ponce   
2745  SJU    Luis Munoz Marin International Airport          San Juan   

                       country  
1902            American Samoa  
2149  Northern Mariana Islands  
2151                      Guam  
2738            Virgin Islands  
2739            Virgin Islands  
2740               Puerto Rico  
2744               Puerto Rico  
2745               Puerto Rico  


In [41]:
US_AND_TERRITORIES = ["United States", "Puerto Rico", "Virgin Islands", "Guam",
                       "American Samoa", "Northern Mariana Islands"]

airports_lookup = airports_raw[
    (airports_raw["iata"].isin(airport_codes)) & (airports_raw["country"].isin(US_AND_TERRITORIES))
][["iata", "name", "city"]].copy()

# Manual patch for the 4 still missing from OpenFlights
manual_airports = pd.DataFrame([
    {"iata": "EAR", "name": "Kearney Regional Airport", "city": "Kearney"},
    {"iata": "IFP", "name": "Laughlin/Bullhead International Airport", "city": "Bullhead City"},
    {"iata": "TKI", "name": "Collin County Regional Airport", "city": "McKinney"},
    {"iata": "XWA", "name": "Williston Basin International Airport", "city": "Williston"},
])

airports_lookup = pd.concat([airports_lookup, manual_airports], ignore_index=True)

print(f"Final count: {len(airports_lookup)} out of {len(airport_codes)} airport codes")
missing_final = airport_codes - set(airports_lookup["iata"])
print(f"Still missing: {sorted(missing_final)}")

Final count: 395 out of 395 airport codes
Still missing: []


In [42]:
airports_lookup.to_csv("../data/processed/dim_airport.csv", index=False)

carrier_lookup = pd.DataFrame(list(CARRIER_NAMES.items()), columns=["carrier_code", "carrier_name"])
carrier_lookup.to_csv("../data/processed/dim_carrier.csv", index=False)

print(carrier_lookup)
print(f"\nSaved both lookup files.")

   carrier_code         carrier_name
0            9E         Endeavor Air
1            AA    American Airlines
2            AS      Alaska Airlines
3            B6      JetBlue Airways
4            DL      Delta Air Lines
5            EV  ExpressJet Airlines
6            F9    Frontier Airlines
7            G4        Allegiant Air
8            HA    Hawaiian Airlines
9            MQ            Envoy Air
10           NK      Spirit Airlines
11           OH         PSA Airlines
12           OO     SkyWest Airlines
13           QX          Horizon Air
14           UA      United Airlines
15           US           US Airways
16           VX       Virgin America
17           WN   Southwest Airlines
18           YV        Mesa Airlines
19           YX      Republic Airway

Saved both lookup files.


In [43]:
def sql_string(val):
    # Use double-quotes as the SQL string delimiter so we don't have to
    # escape apostrophes in names like "O'Hare International Airport"
    return f'"{val}"'

airport_rows = ",\n  ".join(
    f'STRUCT({sql_string(row.iata)} AS airport_code, {sql_string(row.city)} AS state)'
    for row in airports_lookup.itertuples()
)
# Note: dim_airport's schema only has (airport_code, state) — using city here
# temporarily to check, but we actually want the airport's STATE, not city.

In [44]:
# Get airport -> state mapping from our own data (both Origin and Dest sides)
state_lookup = {}
for f in glob.glob("../data/processed/ontime/*.parquet"):
    df = pd.read_parquet(f, columns=["Origin", "OriginState", "Dest", "DestState"])
    state_lookup.update(dict(zip(df["Origin"], df["OriginState"])))
    state_lookup.update(dict(zip(df["Dest"], df["DestState"])))
    break  # one file has every airport code that'll ever appear, no need to scan all 132

airports_lookup["state"] = airports_lookup["iata"].map(state_lookup)
airports_lookup = airports_lookup.rename(columns={"iata": "airport_code", "name": "airport_name"})

print(airports_lookup.head())
print(f"\nMissing state for: {airports_lookup['state'].isna().sum()} airports")

  airport_code                              airport_name              city  \
0          PPG           Pago Pago International Airport         Pago Pago   
1          SPN              Saipan International Airport            Saipan   
2          GUM  Antonio B. Won Pat International Airport             Agana   
3          STT                     Cyril E. King Airport        St. Thomas   
4          STX                   Henry E Rohlsen Airport  St. Croix Island   

  state  
0    TT  
1   NaN  
2    TT  
3    VI  
4    VI  

Missing state for: 83 airports


In [45]:
state_lookup = {}
for f in glob.glob("../data/processed/ontime/*.parquet"):
    df = pd.read_parquet(f, columns=["Origin", "OriginState", "Dest", "DestState"])
    state_lookup.update(dict(zip(df["Origin"], df["OriginState"])))
    state_lookup.update(dict(zip(df["Dest"], df["DestState"])))

airports_lookup["state"] = airports_lookup["airport_code"].map(state_lookup)

print(f"Missing state for: {airports_lookup['state'].isna().sum()} airports")
print(airports_lookup[airports_lookup["state"].isna()])

Missing state for: 0 airports
Empty DataFrame
Columns: [airport_code, airport_name, city, state]
Index: []


In [46]:
def sql_str(val):
    if pd.isna(val):
        return "NULL"
    escaped = str(val).replace('"', '\\"')  # escape any literal double-quotes in names
    return f'"{escaped}"'

# Build the dim_airport INSERT-as-CREATE statement
airport_rows = ",\n  ".join(
    f'STRUCT({sql_str(r.airport_code)} AS airport_code, {sql_str(r.airport_name)} AS airport_name, '
    f'{sql_str(r.city)} AS city, {sql_str(r.state)} AS state)'
    for r in airports_lookup.itertuples()
)

airport_sql = f"""CREATE OR REPLACE TABLE `airline-delay-platform.airline_data.dim_airport` AS
SELECT * FROM UNNEST([
  {airport_rows}
]);"""

with open("../sql/dim_airport.sql", "w") as f:
    f.write(airport_sql)

print(f"dim_airport.sql written, {len(airports_lookup)} rows")
print(f"First 500 characters:\n{airport_sql[:500]}")

dim_airport.sql written, 395 rows
First 500 characters:
CREATE OR REPLACE TABLE `airline-delay-platform.airline_data.dim_airport` AS
SELECT * FROM UNNEST([
  STRUCT("PPG" AS airport_code, "Pago Pago International Airport" AS airport_name, "Pago Pago" AS city, "TT" AS state),
  STRUCT("SPN" AS airport_code, "Saipan International Airport" AS airport_name, "Saipan" AS city, "TT" AS state),
  STRUCT("GUM" AS airport_code, "Antonio B. Won Pat International Airport" AS airport_name, "Agana" AS city, "TT" AS state),
  STRUCT("STT" AS airport_code, "Cyril E.


In [47]:
carrier_rows = ",\n  ".join(
    f'STRUCT({sql_str(r.carrier_code)} AS carrier_code, {sql_str(r.carrier_name)} AS carrier_name)'
    for r in carrier_lookup.itertuples()
)

carrier_sql = f"""CREATE OR REPLACE TABLE `airline-delay-platform.airline_data.dim_carrier` AS
SELECT * FROM UNNEST([
  {carrier_rows}
]);"""

with open("../sql/dim_carrier.sql", "w") as f:
    f.write(carrier_sql)

print(f"dim_carrier.sql written, {len(carrier_lookup)} rows")
print(carrier_sql)  # small enough to print in full

dim_carrier.sql written, 20 rows
CREATE OR REPLACE TABLE `airline-delay-platform.airline_data.dim_carrier` AS
SELECT * FROM UNNEST([
  STRUCT("9E" AS carrier_code, "Endeavor Air" AS carrier_name),
  STRUCT("AA" AS carrier_code, "American Airlines" AS carrier_name),
  STRUCT("AS" AS carrier_code, "Alaska Airlines" AS carrier_name),
  STRUCT("B6" AS carrier_code, "JetBlue Airways" AS carrier_name),
  STRUCT("DL" AS carrier_code, "Delta Air Lines" AS carrier_name),
  STRUCT("EV" AS carrier_code, "ExpressJet Airlines" AS carrier_name),
  STRUCT("F9" AS carrier_code, "Frontier Airlines" AS carrier_name),
  STRUCT("G4" AS carrier_code, "Allegiant Air" AS carrier_name),
  STRUCT("HA" AS carrier_code, "Hawaiian Airlines" AS carrier_name),
  STRUCT("MQ" AS carrier_code, "Envoy Air" AS carrier_name),
  STRUCT("NK" AS carrier_code, "Spirit Airlines" AS carrier_name),
  STRUCT("OH" AS carrier_code, "PSA Airlines" AS carrier_name),
  STRUCT("OO" AS carrier_code, "SkyWest Airlines" AS carrier_name),

In [49]:
# Aggregate on-time data up to route/carrier/quarter grain first
sample_ontime = pd.read_parquet("../data/processed/ontime/ontime_2015_01.parquet")

route_summary = sample_ontime.groupby(
    ["Year", "Quarter", "Reporting_Airline", "Origin", "Dest"]
).agg(
    total_flights=("FlightDate", "count"),
    avg_dep_delay=("DepDelay", "mean"),
    pct_delayed_15=("DepDel15", "mean"),
).reset_index()

print(f"Route-carrier-quarter combinations in Jan 2015 alone: {len(route_summary):,}")
print(route_summary.head())

Route-carrier-quarter combinations in Jan 2015 alone: 6,525
   Year  Quarter Reporting_Airline Origin Dest  total_flights  avg_dep_delay  \
0  2015        1                AA    ABQ  DFW            124       4.715447   
1  2015        1                AA    ATL  DFW            283      10.156028   
2  2015        1                AA    ATL  MIA            114       3.517544   
3  2015        1                AA    AUS  DFW            417       7.511057   
4  2015        1                AA    AUS  JFK             31       4.357143   

   pct_delayed_15  
0        0.081301  
1        0.191489  
2        0.114035  
3        0.137592  
4        0.142857  


In [50]:
fare_q1_2015 = pd.read_parquet("../data/processed/db1b/db1b_2015_q1.parquet")

# Aggregate fares to the same grain: route + carrier + quarter
fare_summary = fare_q1_2015.groupby(
    ["Year", "Quarter", "OpCarrier", "Origin", "Dest"]
).agg(
    avg_fare=("MktFare", "mean"),
    total_passengers=("Passengers", "sum"),
).reset_index()

print(f"Route-carrier-quarter combinations in fare data (Q1 2015): {len(fare_summary):,}")

# Now the actual join — note we're joining our January-only flight summary
# against a FULL QUARTER of fare data, so a lot of mismatch is expected here
# (that's a join-grain issue, not a bug) — more on that below
joined = route_summary.merge(
    fare_summary,
    left_on=["Year", "Quarter", "Reporting_Airline", "Origin", "Dest"],
    right_on=["Year", "Quarter", "OpCarrier", "Origin", "Dest"],
    how="left",
)

matched = joined["avg_fare"].notna().sum()
print(f"\nRoutes with a matching fare: {matched:,} out of {len(joined):,} ({100*matched/len(joined):.1f}%)")
print(joined[["Origin", "Dest", "Reporting_Airline", "total_flights", "avg_dep_delay", "avg_fare"]].head(10))

Route-carrier-quarter combinations in fare data (Q1 2015): 142,534

Routes with a matching fare: 6,447 out of 6,525 (98.8%)
  Origin Dest Reporting_Airline  total_flights  avg_dep_delay    avg_fare
0    ABQ  DFW                AA            124       4.715447  256.077273
1    ATL  DFW                AA            283      10.156028  209.186945
2    ATL  MIA                AA            114       3.517544  169.907573
3    AUS  DFW                AA            417       7.511057  202.993270
4    AUS  JFK                AA             31       4.357143  277.472834
5    AUS  LAX                AA            116       2.256637  303.564868
6    AUS  ORD                AA             90       4.511628  277.890828
7    BDL  DFW                AA             63      18.982759  407.472394
8    BDL  MIA                AA             31       9.413793  210.025233
9    BHM  DFW                AA             58      37.896552  233.820437


In [51]:
unmatched = joined[joined["avg_fare"].isna()]
print(f"Unmatched routes: {len(unmatched)}")
print(unmatched[["Origin", "Dest", "Reporting_Airline", "total_flights", "avg_dep_delay"]].sort_values("total_flights", ascending=False).head(15))

Unmatched routes: 78
     Origin Dest Reporting_Airline  total_flights  avg_dep_delay
2349    MEI  PIB                EV             53       2.150943
3954    JMS  DVL                OO             48      24.731707
2538    PIB  MEI                EV             31      20.838710
3875    HIB  INL                OO             27       2.592593
3942    INL  HIB                OO             27      14.740741
3935    IMT  RHI                OO             27      39.153846
3812    DVL  JMS                OO             27      49.130435
4294    RHI  IMT                OO             26       5.269231
3991    LAX  SAF                OO             19      -2.352941
4303    SAF  LAX                OO             19       9.176471
4015    MBS  DTW                OO             18      15.166667
3573    AZO  DTW                OO             13      23.000000
2271    LAN  DTW                EV              7      21.714286
1977    DTW  LAN                EV              7      19.000000
4580